In [ ]:
import numpy as np
from scipy.special import erf  # нужен для GELU, руками считать не стал — там интеграл от гауссианы


**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [ ]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # простейший случай — просто возвращаем вход

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # простейший случай — просто возвращаем вход

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential — контейнер для последовательных слоёв

**Define** a forward and backward pass procedures.

In [ ]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []
        self._inputs = []  # храним входы каждого слоя — без этого backward не работает корректно

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        """
        Basic workflow of FORWARD PASS:

            y_0    = module[0].forward(input)
            y_1    = module[1].forward(y_0)
            ...
            output = module[n-1].forward(y_{n-2})

        Store intermediate inputs for use in backward.
        """
        # сначала кладём исходный вход, потом выход каждого слоя
        # в итоге _inputs[i] — это то что видел i-й модуль на входе
        self._inputs = [input]
        current = input
        for module in self.modules:
            current = module.forward(current)
            self._inputs.append(current)
        self.output = current
        return self.output

    def backward(self, input, gradOutput):
        """
        Workflow of BACKWARD PASS:

            g_{n-1} = module[n-1].backward(y_{n-2}, gradOutput)
            g_{n-2} = module[n-2].backward(y_{n-3}, g_{n-1})
            ...
            gradInput = module[0].backward(input, g_1)

        !!!

        To each module you need to provide the input, module saw while forward pass,
        it is used while computing gradients.
        Make sure that the input for `i-th` layer the output of `module[i]` (just the same input as in forward pass)
        and NOT `input` to this Sequential module.

        !!!

        """
        # идём с конца — каждый слой получает именно свой вход, не общий input контейнера
        # сначала долго думал почему _inputs[i], а не просто input — потом дошло
        grad = gradOutput
        for i in reversed(range(len(self.modules))):
            grad = self.modules[i].backward(self._inputs[i], grad)
        self.gradInput = grad
        return self.gradInput


    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        """
        Should gather all parameters in a list.
        """
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        """
        Should gather all gradients w.r.t parameters in a list.
        """
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        """
        Propagates training parameter through all modules
        """
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        """
        Propagates training parameter through all modules
        """
        self.training = False
        for module in self.modules:
            module.evaluate()


# Слои

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [ ]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # Xavier-подобная инициализация — дисперсия ~1/n_in
        # пробовал нули — сеть не обучалась вообще, все градиенты одинаковые
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = input @ self.W.T + self.b  # y = xW^T + b
        return self.output

    def updateGradInput(self, input, gradOutput):
        # dL/dx = dL/dy * W — размерности: (bs, n_out) @ (n_out, n_in) = (bs, n_in)
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        # dL/dW = dL/dyᵀ * x — суммируем по батчу автоматически через матумножение
        self.gradW = gradOutput.T @ input
        # bias: градиент просто сумма по батчу (bias одинаковый для всех примеров)
        self.gradb = gradOutput.sum(axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q


## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [ ]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # вычитаем max перед exp — иначе exp(большое число) = inf
        # математически результат тот же: softmax(x) = softmax(x - c)
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        self.output = np.exp(self.output)
        self.output = self.output / self.output.sum(axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # якобиан softmax: J_ij = s_i*(delta_ij - s_j)
        # после свёртки с gradOutput получаем: grad_i = s_i*(g_i - sum_j(g_j*s_j))
        s = self.output
        dot = (gradOutput * s).sum(axis=1, keepdims=True)
        self.gradInput = s * (gradOutput - dot)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"


## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [ ]:
class LogSoftMax(Module):
    # LogSoftMax нужен чтобы NLLCriterion не брал log внутри себя
    # комбо LogSoftMax + ClassNLLCriterion = стабильный cross-entropy
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # вычитаем max для стабильности
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        # log(softmax(x)) = x_shifted - log(sum(exp(x_shifted)))
        log_sum_exp = np.log(np.exp(self.output).sum(axis=1, keepdims=True))
        self.output = self.output - log_sum_exp
        return self.output

    def updateGradInput(self, input, gradOutput):
        # output уже log_softmax, поэтому exp(output) = softmax — не пересчитываем
        s = np.exp(self.output)
        # d(log_sm_i)/dx_j = delta_ij - sm_j => grad_i = g_i - sm_i * sum(g)
        self.gradInput = gradOutput - s * gradOutput.sum(axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"


## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [ ]:
class BatchNormalization(Module):
    EPS = 1e-3  # 1e-5 было бы точнее, но при маленьких батчах дисперсия скачет — лучше побольше
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        if self.training:
            batch_mean = input.mean(axis=0)
            batch_var  = input.var(axis=0)

            # обновляем EMA
            # инициализируем EMA с первого батча, потом обновляем
            if self.moving_mean is None:
                self.moving_mean     = batch_mean.copy()
                self.moving_variance = batch_var.copy()
            else:
                self.moving_mean     = self.moving_mean     * self.alpha + batch_mean * (1 - self.alpha)
                self.moving_variance = self.moving_variance * self.alpha + batch_var  * (1 - self.alpha)

            # кэшируем для backward
            self._batch_mean = batch_mean
            self._batch_var  = batch_var

            self.output = (input - batch_mean) / np.sqrt(batch_var + self.EPS)
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)
        return self.output

    def updateGradInput(self, input, gradOutput):
        N   = input.shape[0]
        mu  = self._batch_mean
        s2  = self._batch_var
        std = np.sqrt(s2 + self.EPS)
        xhat = (input - mu) / std

        # формула backward из статьи Ioffe & Szegedy (2015)
        # долго выводил руками через цепное правило — в итоге сошлось
        # три слагаемых: прямой градиент, поправка на среднее, поправка на дисперсию
        self.gradInput = (1.0 / (N * std)) * (
            N * gradOutput
            - gradOutput.sum(axis=0)
            - xhat * (gradOutput * xhat).sum(axis=0)
        )
        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"


In [ ]:
class ChannelwiseScaling(Module):
    # поэлементное масштабирование y = γ·x + β
    # γ и β — обучаемые параметры, идут после BatchNormalization
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [ ]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()
        self.p = p
        self.mask = None

    def updateOutput(self, input):
        if not self.training:
            self.output = input.copy()
            return self.output
        # p=1 или p=0 — граничные случаи, без проверки будет деление на ноль
        if self.p >= 1.0:
            self.mask = np.zeros_like(input)
            self.output = np.zeros_like(input)
            return self.output
        if self.p <= 0.0:
            self.mask = np.ones_like(input)
            self.output = input.copy()
            return self.output
        # inverted dropout — делим на (1-p) чтобы E[output] = E[input] и в тесте ничего не менять
        self.mask = (np.random.uniform(0, 1, input.shape) > self.p).astype(input.dtype)
        self.output = input * self.mask / (1.0 - self.p)
        return self.output

    def updateGradInput(self, input, gradOutput):
        if not self.training:
            self.gradInput = gradOutput.copy()
            return self.gradInput
        if self.p >= 1.0:
            self.gradInput = np.zeros_like(gradOutput)
            return self.gradInput
        if self.p <= 0.0:
            self.gradInput = gradOutput.copy()
            return self.gradInput
        self.gradInput = gradOutput * self.mask / (1.0 - self.p)
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [ ]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride       = stride      if isinstance(stride,      tuple) else (stride,      stride)
        self.padding      = padding     if isinstance(padding,     tuple) else (padding,     padding)
        self.use_bias     = bias        # boolean flag; self.bias holds the actual weight array
        self.padding_mode = padding_mode

        # веса устанавливаются снаружи (например из тестов)
        self.weight    = None
        self.bias      = None
        self.gradWeight = None
        self.gradBias   = None

    # im2col: раскладываем все патчи входа в строки матрицы
    # тогда свёртка = обычное матричное умножение, numpy делает это быстро
    # TODO: переписать _im2col через as_strided — будет быстрее без копий памяти
    def _im2col(self, x_pad, kH, kW, sH, sW, out_H, out_W):
        N, C, _, _ = x_pad.shape
        col = np.zeros((N, out_H * out_W, C * kH * kW), dtype=x_pad.dtype)
        for i in range(out_H):
            for j in range(out_W):
                col[:, i * out_W + j, :] = (
                    x_pad[:, :, i*sH : i*sH+kH, j*sW : j*sW+kW]
                    .reshape(N, -1)
                )
        return col   # (N, out_H*out_W, C*kH*kW)

    # col2im: обратная операция — складываем градиенты обратно по патчам
    # важно именно += а не = из-за перекрывающихся окон при stride < kernel
    def _col2im(self, dcol, N, C, H, W, kH, kW, sH, sW, pH, pW, out_H, out_W):
        dx_pad = np.zeros((N, C, H + 2*pH, W + 2*pW), dtype=dcol.dtype)
        for i in range(out_H):
            for j in range(out_W):
                dx_pad[:, :, i*sH : i*sH+kH, j*sW : j*sW+kW] += (
                    dcol[:, :, i * out_W + j].reshape(N, C, kH, kW)
                )
        return dx_pad   # (N, C, H_pad, W_pad)

    def updateOutput(self, input):
        N, C_in, H, W = input.shape
        kH, kW = self.kernel_size
        sH, sW = self.stride
        pH, pW = self.padding

        # добавляем паддинг если нужен
        if pH > 0 or pW > 0:
            x_pad = np.pad(input, ((0,0),(0,0),(pH,pH),(pW,pW)), mode='constant')
        else:
            x_pad = input
        self._x_pad = x_pad

        out_H = (H + 2*pH - kH) // sH + 1
        out_W = (W + 2*pW - kW) // sW + 1

        col   = self._im2col(x_pad, kH, kW, sH, sW, out_H, out_W)
        # col: (N, out_H*out_W, C_in*kH*kW) — каждая строка = один патч

        W_mat = self.weight.reshape(self.out_channels, -1)  # (C_out, C_in*kH*kW)
        out   = col @ W_mat.T                               # (N, out_H*out_W, C_out)
        out   = out.transpose(0, 2, 1).reshape(N, self.out_channels, out_H, out_W)

        if self.use_bias and self.bias is not None:
            out = out + self.bias[None, :, None, None]

        self.output = out
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C_in, H, W = input.shape
        kH, kW = self.kernel_size
        sH, sW = self.stride
        pH, pW = self.padding
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        W_mat    = self.weight.reshape(self.out_channels, -1)  # (C_out, C_in*kH*kW)
        grad_out = gradOutput.reshape(N, self.out_channels, out_H * out_W)
        # (N, C_in*kH*kW, out_H*out_W)
        dcol = np.tensordot(W_mat.T, grad_out, axes=([1], [1]))  # (C_in*kH*kW, N, out_H*out_W)
        dcol = dcol.transpose(1, 0, 2)                           # (N, C_in*kH*kW, out_H*out_W)

        dx_pad = self._col2im(dcol, N, C_in, H, W, kH, kW, sH, sW, pH, pW, out_H, out_W)

        if pH > 0 or pW > 0:
            self.gradInput = dx_pad[:, :, pH : pH+H, pW : pW+W]
        else:
            self.gradInput = dx_pad
        return self.gradInput

    def __repr__(self):
        return "Conv2d"


#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [ ]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding=0):
        super(MaxPool2d, self).__init__()
        self.kernel_size = kernel_size
        self.stride      = stride
        self.padding     = padding

    def updateOutput(self, input):
        N, C, H, W = input.shape
        kH = kW = self.kernel_size
        sH = sW = self.stride
        pH = pW = self.padding

        if pH > 0:
            x_pad = np.pad(input, ((0,0),(0,0),(pH,pH),(pW,pW)),
                           constant_values=-np.inf)
        else:
            x_pad = input

        out_H = (H + 2*pH - kH) // sH + 1
        out_W = (W + 2*pW - kW) // sW + 1

        self.output  = np.zeros((N, C, out_H, out_W), dtype=input.dtype)
        # сохраняем где именно был максимум — в backward нужно туда вернуть градиент
        # маски тоже можно, но тут нагляднее и быстрее
        self._max_i  = np.zeros((N, C, out_H, out_W), dtype=np.int64)
        self._max_j  = np.zeros((N, C, out_H, out_W), dtype=np.int64)
        self._x_pad  = x_pad
        self._input_shape = input.shape
        self._pad    = (pH, pW)

        for oh in range(out_H):
            for ow in range(out_W):
                patch = x_pad[:, :, oh*sH : oh*sH+kH, ow*sW : ow*sW+kW]
                # argmax по двум пространственным осям — находим позицию максимума
                flat  = patch.reshape(N, C, -1)
                idx   = flat.argmax(axis=2)           # (N, C)
                ih    = oh*sH + idx // kW
                iw    = ow*sW + idx %  kW
                self._max_i[:, :, oh, ow] = ih
                self._max_j[:, :, oh, ow] = iw
                self.output[:, :, oh, ow] = x_pad[
                    np.arange(N)[:,None], np.arange(C)[None,:], ih, iw
                ]
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self._input_shape
        pH, pW     = self._pad
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        dx_pad = np.zeros_like(self._x_pad)

        for oh in range(out_H):
            for ow in range(out_W):
                ih = self._max_i[:, :, oh, ow]   # (N, C)
                iw = self._max_j[:, :, oh, ow]
                # np.add.at — атомарный scatter-add
                # нельзя просто dx_pad[..., ih, iw] = grad — при перекрытии окон перезапишет
                np.add.at(dx_pad,
                          (np.arange(N)[:,None], np.arange(C)[None,:], ih, iw),
                          gradOutput[:, :, oh, ow])

        if pH > 0:
            self.gradInput = dx_pad[:, :, pH:pH+H, pW:pW+W]
        else:
            self.gradInput = dx_pad
        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    # среднее по окну вместо максимума — мягче, градиент равномерно распределяется
    def __init__(self, kernel_size, stride, padding=0):
        super(AvgPool2d, self).__init__()
        self.kernel_size = kernel_size
        self.stride      = stride
        self.padding     = padding

    def updateOutput(self, input):
        N, C, H, W = input.shape
        kH = kW = self.kernel_size
        sH = sW = self.stride
        pH = pW = self.padding

        # паддинг нулями (не -inf как в MaxPool)
        if pH > 0:
            x_pad = np.pad(input, ((0,0),(0,0),(pH,pH),(pW,pW)), mode='constant')
        else:
            x_pad = input

        out_H = (H + 2*pH - kH) // sH + 1
        out_W = (W + 2*pW - kW) // sW + 1

        self.output = np.zeros((N, C, out_H, out_W), dtype=input.dtype)
        self._input_shape = input.shape
        self._pad = (pH, pW)

        for oh in range(out_H):
            for ow in range(out_W):
                patch = x_pad[:, :, oh*sH : oh*sH+kH, ow*sW : ow*sW+kW]
                self.output[:, :, oh, ow] = patch.mean(axis=(-2, -1))
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self._input_shape
        pH, pW = self._pad
        kH = kW = self.kernel_size
        sH = sW = self.stride
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        dx_pad = np.zeros((N, C, H + 2*pH, W + 2*pW), dtype=input.dtype)

        # градиент равномерно распределяется по всем элементам окна
        for oh in range(out_H):
            for ow in range(out_W):
                dx_pad[:, :, oh*sH : oh*sH+kH, ow*sW : ow*sW+kW] += (
                    gradOutput[:, :, oh, ow][:, :, None, None] / (kH * kW)
                )

        if pH > 0:
            self.gradInput = dx_pad[:, :, pH:pH+H, pW:pW+W]
        else:
            self.gradInput = dx_pad
        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"


#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [ ]:
class GlobalMaxPool2d(Module):
    # агрегируем (H×W) → (1×1) по максимуму в каждом канале
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()

    def updateOutput(self, input):
        self.output = input.max(axis=(-2, -1), keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        max_val = input.max(axis=(-2, -1), keepdims=True)
        mask    = (input == max_val).astype(float)
        mask   /= mask.sum(axis=(-2, -1), keepdims=True)
        self.gradInput = mask * gradOutput
        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"


class GlobalAvgPool2d(Module):
    # агрегируем (H×W) → (1×1) по среднему в каждом канале
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()

    def updateOutput(self, input):
        self.output = input.mean(axis=(-2, -1), keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        H, W = input.shape[-2], input.shape[-1]
        self.gradInput = gradOutput * np.ones_like(input) / (H * W)
        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"


#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [ ]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim   = end_dim

    def updateOutput(self, input):
        ndim  = input.ndim
        start = self.start_dim if self.start_dim >= 0 else ndim + self.start_dim
        end   = self.end_dim   if self.end_dim   >= 0 else ndim + self.end_dim
        new_shape = input.shape[:start] + (-1,) + input.shape[end + 1:]
        self.output = input.reshape(new_shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"


# Функции активации

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [ ]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [ ]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input >= 0, input, self.slope * input)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input >= 0, 1.0, self.slope)
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"


## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [ ]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input >= 0, input, self.alpha * (np.exp(input) - 1.0))
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input >= 0, 1.0, self.alpha * np.exp(input))
        return self.gradInput

    def __repr__(self):
        return "ELU"


## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [ ]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        # log(1+exp(x)) взрывается при больших x — используем тождество:
        # log(1+e^x) = log(1+e^{-|x|}) + max(x,0)  — оба слагаемых ограничены
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # производная log(1+e^x) = sigmoid(x)
        sig = 1.0 / (1.0 + np.exp(-input))
        self.gradInput = gradOutput * sig
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"


#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [ ]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()  # была опечатка с SoftPlus, поправил

    def updateOutput(self, input):
        # GELU(x) = x * Phi(x), Phi — CDF стандартного нормального распределения
        # используем erf из scipy, вручную через интеграл не стал — смысла нет
        self.output = input * 0.5 * (1.0 + erf(input / np.sqrt(2.0)))
        return self.output

    def updateGradInput(self, input, gradOutput):
        # производная: d/dx [x*Phi(x)] = Phi(x) + x*phi(x)  по правилу произведения
        # phi = Phi' — плотность нормального распределения
        Phi = 0.5 * (1.0 + erf(input / np.sqrt(2.0)))
        phi = np.exp(-0.5 * input**2) / np.sqrt(2.0 * np.pi)
        self.gradInput = gradOutput * (Phi + input * phi)
        return self.gradInput

    def __repr__(self):
        return "Gelu"


# Функции потерь

Criterions are used to score the models answers.

In [ ]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [ ]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [ ]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):
        # клипаем вероятности чтоб не было log(0) = -inf
        # "unstable" потому что если входы большие — softmax уже теряет точность до этого
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        # клипаем чтоб не было деления на 0
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        # dNLL/dp_i = -target_i / (p_i * N)
        self.gradInput = -target / (input_clamp * input.shape[0])
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"


## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [ ]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        # input должен быть log-вероятностями (выход LogSoftMax), не сырыми логитами
        assert input.shape == target.shape, f"input {input.shape} != target {target.shape}"
        # NLL = -mean( sum_j target_j * log_p_j ) — для one-hot target просто выбираем нужный столбец
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        # dNLL/d(log_p_i) = -target_i / N
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"


1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

## Дополнительные модули для декодера
`Reshape`, `Upsample2d`, `Sigmoid` — нужны для декодера свёрточного автоэнкодера.

In [ ]:
class Reshape(Module):
    # меняем форму (N, flat) → (N, C, H, W)
    # нужен в декодере чтобы из вектора вернуться к feature maps
    def __init__(self, *shape):
        super(Reshape, self).__init__()
        self.shape = shape  # целевая форма без batch dim

    def updateOutput(self, input):
        self._in_shape = input.shape
        self.output = input.reshape((input.shape[0],) + self.shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(self._in_shape)
        return self.gradInput

    def __repr__(self):
        return f"Reshape{self.shape}"


class Upsample2d(Module):
    # nearest-neighbour upsampling — увеличиваем пространство в factor раз
    def __init__(self, factor):
        super(Upsample2d, self).__init__()
        self.factor = factor

    def updateOutput(self, input):
        f = self.factor
        self.output = np.repeat(np.repeat(input, f, axis=2), f, axis=3)
        return self.output

    def updateGradInput(self, input, gradOutput):
        f = self.factor
        # суммируем градиенты внутри каждого блока f×f
        N, C, H, W = input.shape
        g = gradOutput.reshape(N, C, H, f, W, f)
        self.gradInput = g.sum(axis=(3, 5))
        return self.gradInput

    def __repr__(self):
        return f"Upsample2d(x{self.factor})"


class Sigmoid(Module):
    # сигмоида — выход в [0,1], нужна в декодере для пикселей
    def __init__(self):
        super(Sigmoid, self).__init__()

    def updateOutput(self, input):
        # для x≥0: 1/(1+e^{-x}), для x<0: e^x/(1+e^x) — оба ограничены
        self.output = np.where(
            input >= 0,
            1.0 / (1.0 + np.exp(-input)),
            np.exp(input) / (1.0 + np.exp(input))
        )
        return self.output

    def updateGradInput(self, input, gradOutput):
        s = self.output
        self.gradInput = gradOutput * s * (1.0 - s)
        return self.gradInput

    def __repr__(self):
        return "Sigmoid"

## Дополнительные функции потерь
`MSE4D` — лосс для 4D-тензоров (автоэнкодер), `MAECriterion` — средняя абсолютная ошибка.

In [ ]:
class MSE4D(Criterion):
    # MSE для 4D — нужен для автоэнкодера где выход (N,C,H,W)
    # обычный MSECriterion делит на N, тут делим на все элементы
    def __init__(self):
        super(MSE4D, self).__init__()

    def updateOutput(self, input, target):
        assert input.shape == target.shape, "MSE4D: input и target должны быть одинаковой формы"
        self.output = float(np.mean((input - target) ** 2))
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = 2.0 * (input - target) / input.size
        return self.gradInput

    def __repr__(self):
        return "MSE4D"


class MAECriterion(Criterion):
    # MAE — средняя абсолютная ошибка, устойчивее к выбросам чем MSE
    def __init__(self):
        super(MAECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = float(np.mean(np.abs(input - target)))
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = np.sign(input - target) / input.size
        return self.gradInput

    def __repr__(self):
        return "MAECriterion"